In [124]:
import pandas as pd
import chainladder as cl
import numpy as np

In [125]:
# Page 140, Exhibit I Sheet 1 Columns 1-2
auto_bi = cl.load_sample("friedland_auto_bi_insurer")
auto_bi["Reported Claims"].latest_diagonal

,2008
2000,"10,000,000"
2001,"8,000,000"
2002,"9,400,000"
2003,"15,600,000"
2004,"16,500,000"
2005,"18,500,000"
2006,"16,500,000"
2007,"14,000,000"
2008,"8,700,000"


In [126]:
# Page 140, Exhibit I Sheet 1 Column 3
auto_bi["Paid Claims"].latest_diagonal

,2008
2000,"9,500,000"
2001,"7,200,000"
2002,"7,600,000"
2003,"7,800,000"
2004,"11,200,000"
2005,"10,200,000"
2006,"6,000,000"
2007,"3,000,000"
2008,"750,000"


In [127]:
# Page 140, Exhibit I Sheet 1 Columns 4-5
# Direct age(in months): value representation per LOB
reported_pattern = {12: 4.0, 24: 2.9, 36: 1.8, 48: 1.4, 60: 1.2, 72: 1.1, 84: 1.03, 96: 1.02, 108: 1.005}
paid_pattern = {12: 90.0, 24: 15.0, 36: 5.0, 48: 2.5, 60: 1.75, 72: 1.35, 84: 1.25, 96: 1.15, 108: 1.05}

In [128]:
# Page 140, Exhibit I Sheet 1 Column 6
Reported_BI = cl.DevelopmentConstant(patterns=reported_pattern, style='cdf').fit_transform(auto_bi["Reported Claims"])
reported_ultimate = cl.Chainladder().fit(Reported_BI).ultimate_
reported_ultimate

,2261
2000,"10,050,000"
2001,"8,160,000"
2002,"9,682,000"
2003,"17,160,000"
2004,"19,800,000"
2005,"25,900,000"
2006,"29,700,000"
2007,"40,600,000"
2008,"34,800,000"


In [129]:
# Page 140, Exhibit I Sheet 1 Column 7
Paid_BI = cl.DevelopmentConstant(patterns=paid_pattern, style='cdf').fit_transform(auto_bi["Paid Claims"])
paid_ultimate = cl.Chainladder().fit(Paid_BI).ultimate_
paid_ultimate

,2261
2000,"9,975,000"
2001,"8,280,000"
2002,"9,500,000"
2003,"10,530,000"
2004,"19,600,000"
2005,"25,500,000"
2006,"30,000,000"
2007,"45,000,000"
2008,"67,500,000"


In [130]:
# Page 140, Exhibit I Sheet 1 Column 8
inital_selected_ultiamte_claims = (reported_ultimate + paid_ultimate)/2
inital_selected_ultiamte_claims

,2261
2000,"10,012,500"
2001,"8,220,000"
2002,"9,591,000"
2003,"13,845,000"
2004,"19,700,000"
2005,"25,700,000"
2006,"29,850,000"
2007,"42,800,000"
2008,"51,150,000"


In [131]:
# Page 140, Exhibit I Sheet 1 Column 9
auto_bi["Earned Premium"].latest_diagonal

,2008
2000,"24,000,000"
2001,"18,000,000"
2002,"19,000,000"
2003,"23,000,000"
2004,"32,000,000"
2005,"47,000,000"
2006,"50,000,000"
2007,"57,000,000"
2008,"62,000,000"


In [132]:
# Page 140, Exhibit I Sheet 1 Column 10
trend_factors = np.round(cl.Trend(trends=[0.145], dates=[("2008-12-31", "2000-01-01")]).fit(auto_bi["Earned Premium"]).trend_.latest_diagonal,3)
trend_factors

,2008
2000,2.9540
2001,2.5800
2002,2.2530
2003,1.9680
2004,1.7190
2005,1.5010
2006,1.3110
2007,1.1450
2008,1.0000


In [133]:
# Page 140, Exhibit I Sheet 1 Column 11
tort_factors = [0.670, 0.670, 0.670, 0.670, 0.750, 1.0, 1.0, 1.0, 1.0]
tort_factors

[0.67, 0.67, 0.67, 0.67, 0.75, 1.0, 1.0, 1.0, 1.0]

In [134]:
# Page 140, Exhibit I Sheet 1 Column 12
trended_adj_ultimate_claims = trend_factors * inital_selected_ultiamte_claims * np.array(tort_factors).reshape(1, 1, -1, 1)
trended_adj_ultimate_claims

,2008
2000,"19,816,540"
2001,"14,209,092"
2002,"14,477,710"
2003,"18,255,463"
2004,"25,398,225"
2005,"38,575,700"
2006,"39,133,350"
2007,"49,006,000"
2008,"51,150,000"


In [135]:
# Page 140, Exhibit I Sheet 1 Column 13
trended_adjusted_claim_ratio = np.round(trended_adj_ultimate_claims/auto_bi["Earned Premium"].latest_diagonal,2)
trended_adjusted_claim_ratio

,2008
2000,0.8300
2001,0.7900
2002,0.7600
2003,0.7900
2004,0.7900
2005,0.8200
2006,0.7800
2007,0.8600
2008,0.8200


In [136]:
# Page 140, Exhibit I Sheet 1 Item 14
print("Average 2000 to 2005:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].mean(),3))
loss_ratios_00_05 = trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_00_05.sum() - loss_ratios_00_05.max() - loss_ratios_00_05.min()) / (len(loss_ratios_00_05) - 2), 3))
print("Average 2001 to 2006:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].mean(),3))
loss_ratios_01_06 = trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_01_06.sum() - loss_ratios_01_06.max() - loss_ratios_01_06.min()) / (len(loss_ratios_01_06) - 2), 3)) # small rounding error due to python's 64 bit float storage

Average 2000 to 2005: 0.797
Average 2000 to 2005 Ex High Ex Low: 0.798
Average 2001 to 2006: 0.788
Average 2000 to 2005 Ex High Ex Low: 0.787


In [137]:
# Page 140, Exhibit I Item 15
selected_claim_ratio = 0.80
selected_claim_ratio

0.8

In [138]:
# Page 140, Exhibit I Item 16
earned_premium = auto_bi["Earned Premium"].loc[:, :, "2008", :].latest_diagonal
exlected_claims_2008 = earned_premium * selected_claim_ratio
exlected_claims_2008

np.float64(49600000.0)

In [139]:
# Page 140, Exhibit I Item 17
print("Total Unpaid Claims:", exlected_claims_2008 - auto_bi["Paid Claims"].loc[:, :, "2008", :].latest_diagonal)
print("Total IBNR:", exlected_claims_2008 - auto_bi["Reported Claims"].loc[:, :, "2008", :].latest_diagonal)

Total Unpaid Claims: 48850000.0
Total IBNR: 40900000.0


In [140]:
# Page 141, Exhibit I Sheet 2, columns 1-2
friedland_gl_self_insurer = cl.load_sample("friedland_gl_self_insurer")
friedland_gl_self_insurer["Reported Claims"].latest_diagonal

,2008
1998,"900,000"
1999,"1,200,000"
2000,"1,300,000"
2001,"1,800,000"
2002,"1,450,000"
2003,"1,400,000"
2004,"2,400,000"
2005,"1,800,000"
2006,"1,500,000"
2007,"1,200,000"


In [141]:
# Page 141, Exhibit I Sheet 2, column 3
friedland_gl_self_insurer["Paid Claims"].latest_diagonal

,2008
1998,"890,000"
1999,"1,170,000"
2000,"1,265,000"
2001,"1,600,000"
2002,"1,200,000"
2003,"1,050,000"
2004,"900,000"
2005,"860,000"
2006,"525,000"
2007,"750,000"


In [142]:
# Page 4, Exhibit I Sheet 2 Columns 4-5
reported_pattern = {12: 3.104, 24: 1.940, 36: 1.616, 48: 1.394, 60: 1.244, 72: 1.131, 84: 1.077, 96: 1.051, 108: 1.030, 120: 1.020, 132: 1.015}
paid_pattern = {12: 20.373, 24: 5.093, 36: 3.183, 48: 2.274, 60: 1.749, 72: 1.489, 84: 1.306, 96: 1.187, 108: 1.109, 120: 1.067, 132: 1.046}

In [143]:
# Page 141, Exhibit I Sheet 2 Columns 6
reported = cl.DevelopmentConstant(patterns=reported_pattern, style='cdf').fit_transform(friedland_gl_self_insurer["Reported Claims"])
reported_ultimate = cl.Chainladder().fit(reported).ultimate_
reported_ultimate

,2261
1998,"913,500"
1999,"1,224,000"
2000,"1,339,000"
2001,"1,891,800"
2002,"1,561,650"
2003,"1,583,400"
2004,"2,985,600"
2005,"2,509,200"
2006,"2,424,000"
2007,"2,328,000"


In [144]:
# Page 141, Exhibit I Sheet 2 Columns 7
paid = cl.DevelopmentConstant(patterns=paid_pattern, style='cdf').fit_transform(friedland_gl_self_insurer["Paid Claims"])
paid_ultimate = cl.Chainladder().fit(paid).ultimate_
paid_ultimate

,2261
1998,"930,940"
1999,"1,248,390"
2000,"1,402,885"
2001,"1,899,200"
2002,"1,567,200"
2003,"1,563,450"
2004,"1,574,100"
2005,"1,955,640"
2006,"1,671,075"
2007,"3,819,750"


In [145]:
# Page 141, Exhibit I Sheet 2 Columns 8
selected_ultimate = (reported_ultimate + paid_ultimate)/2
selected_ultimate

,2261
1998,"922,220"
1999,"1,236,195"
2000,"1,370,942"
2001,"1,895,500"
2002,"1,564,425"
2003,"1,573,425"
2004,"2,279,850"
2005,"2,232,420"
2006,"2,047,538"
2007,"3,073,875"


In [146]:
# Page 141, Exhibit I Sheet 2 Columns 9
population = friedland_gl_self_insurer["Population"].loc[:, :, "2008", :].latest_diagonal
population

,2008
1998,"709,000"
1999,"724,000"
2000,"736,000"
2001,"740,000"
2002,"750,000"
2003,"760,000"
2004,"770,000"
2005,"775,000"
2006,"780,000"
2007,"785,000"


In [147]:
# Page 141, Exhibit I Sheet 2 Column 10
trend_factors = np.round(cl.Trend(trends=[0.075], dates=[("2008-12-31", "1998-01-01")]).fit(friedland_gl_self_insurer["Population"]).trend_.latest_diagonal,3)
trend_factors

,2008
1998,2.0610
1999,1.9170
2000,1.7830
2001,1.6590
2002,1.5430
2003,1.4360
2004,1.3350
2005,1.2420
2006,1.1560
2007,1.0750


In [152]:
# Page 141, Exhibit I Sheet 2 Column 11
trended_adj_ultimate_claims = selected_ultimate * trend_factors
trended_adj_ultimate_claims

,2261
1998,"1,900,695"
1999,"2,369,786"
2000,"2,444,390"
2001,"3,144,634"
2002,"2,413,908"
2003,"2,259,438"
2004,"3,043,600"
2005,"2,772,666"
2006,"2,366,953"
2007,"3,304,416"


In [155]:
# Page 141, Exhibit I Sheet 2 Column 12
trended_pure_premium = np.round(trended_adj_ultimate_claims / population, 2)
trended_pure_premium

,2261
1998,2.6800
1999,3.2700
2000,3.3200
2001,4.2500
2002,3.2200
2003,2.9700
2004,3.9500
2005,3.5800
2006,3.0300
2007,4.2100


In [165]:
# Page 141, Exhibit I Sheet 2 Item 13
print("Average 2000 to 2005:", np.round(trended_pure_premium.iloc[:, :, 2:8, :].mean(),2))
loss_ratios_00_05 = trended_pure_premium.iloc[:, :, 2:8, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_00_05.sum() - loss_ratios_00_05.max() - loss_ratios_00_05.min()) / (len(loss_ratios_00_05) - 2), 2))
print("Average 2001 to 2006:", np.round(trended_pure_premium.iloc[:, :, 3:9, :].mean(),2))
loss_ratios_01_06 = trended_pure_premium.iloc[:, :, 3:9, :].values.flatten()
print("Average 2001 to 2006 Ex High Ex Low:", np.round((loss_ratios_01_06.sum() - loss_ratios_01_06.max() - loss_ratios_01_06.min()) / (len(loss_ratios_01_06) - 2), 2)) # small rounding error due to python's 64 bit float storage

Average 2000 to 2005: 3.55
Average 2000 to 2005 Ex High Ex Low: 3.52
Average 2001 to 2006: 3.5
Average 2001 to 2006 Ex High Ex Low: 3.44


In [167]:
# Page 141, Exhibit I Sheet 2 Item 14
selected_pure_premium = 3.50
selected_pure_premium

3.5

In [186]:
# Page 141, Exhibit I Sheet 2 Item 15

expected_claims_2008 = selected_pure_premium * population.loc[:, :, "2008", :].latest_diagonal
expected_claims_2008

np.float64(2765000.0)

In [ ]:
# Page 141, Exhibit I Item 16
print("Total Unpaid Claims:", exlected_claims_2008 - auto_bi["Paid Claims"].loc[:, :, "2008", :].latest_diagonal)
print("Total IBNR:", exlected_claims_2008 - auto_bi["Reported Claims"].loc[:, :, "2008", :].latest_diagonal)

In [150]:
assert False

AssertionError: 

In [ ]:
# Page 142, Exhibit II Sheet 1, columns 1-2
friedland_us_industry_auto = cl.load_sample("friedland_us_industry_auto")
friedland_us_industry_auto["Reported Claims"].latest_diagonal

,2007
1998,"47,742,304"
1999,"51,185,767"
2000,"54,837,929"
2001,"56,299,562"
2002,"58,592,712"
2003,"57,565,344"
2004,"56,976,657"
2005,"56,786,410"
2006,"54,641,339"
2007,"48,853,563"


In [ ]:
# Page 142, Exhibit II Sheet 1, columns 3
friedland_us_industry_auto["Paid Claims"].latest_diagonal

,2007
1998,"47,644,187"
1999,"51,000,534"
2000,"54,533,225"
2001,"55,878,421"
2002,"57,807,215"
2003,"55,930,654"
2004,"53,774,672"
2005,"50,644,994"
2006,"43,606,497"
2007,"27,229,969"


In [ ]:
# Page 142, Exhibit II Sheet 1, columns 4-5
reported_pattern = {12: 1.292, 24: 1.110, 36: 1.051, 48: 1.023, 60: 1.011, 72: 1.006, 84: 1.003, 96: 1.001, 108: 1.000, 120: 1.000}
paid_pattern = {12: 2.390, 24: 1.404, 36: 1.184, 48: 1.085, 60: 1.040, 72: 1.020, 84: 1.011, 96: 1.006, 108: 1.004, 120: 1.002}

In [ ]:
# Page 142, Exhibit II Sheet 1, column 6
reported_ultimate = cl.Chainladder().fit(cl.DevelopmentConstant(patterns=reported_pattern, style='cdf').fit_transform(friedland_us_industry_auto["Reported Claims"])).ultimate_
reported_ultimate

,2261
1998,"47,742,304"
1999,"51,185,767"
2000,"54,892,767"
2001,"56,468,461"
2002,"58,944,268"
2003,"58,198,563"
2004,"58,287,120"
2005,"59,682,517"
2006,"60,651,886"
2007,"63,118,803"


In [ ]:
# Page 142, Exhibit II Sheet 1, column 7
paid_ultimate = cl.Chainladder().fit(cl.DevelopmentConstant(patterns=paid_pattern, style='cdf').fit_transform(friedland_us_industry_auto["Paid Claims"])).ultimate_
paid_ultimate

,2261
1998,"47,739,475"
1999,"51,204,536"
2000,"54,860,424"
2001,"56,493,084"
2002,"58,963,359"
2003,"58,167,880"
2004,"58,345,519"
2005,"59,963,673"
2006,"61,223,522"
2007,"65,079,626"


In [ ]:
# Page 142, Exhibit II Sheet 1, column 8
selected_ultimate = (reported_ultimate + paid_ultimate)/2
selected_ultimate

,2261
1998,"47,740,890"
1999,"51,195,152"
2000,"54,876,596"
2001,"56,480,772"
2002,"58,953,814"
2003,"58,183,221"
2004,"58,316,320"
2005,"59,823,095"
2006,"60,937,704"
2007,"64,099,215"


In [ ]:
# Page 142, Exhibit II Sheet 1, column 9
earned_premium =friedland_us_industry_auto["Earned Premium"].latest_diagonal
earned_premium

,2007
1998,"68,574,209"
1999,"68,544,981"
2000,"68,907,977"
2001,"72,544,955"
2002,"79,228,887"
2003,"86,643,542"
2004,"91,763,523"
2005,"94,115,312"
2006,"95,272,279"
2007,"95,176,240"


In [ ]:
# Page 142, Exhibit II Sheet 1, column 10
np.round(selected_ultimate / earned_premium, 3)

,2261
1998,0.6960
1999,0.7470
2000,0.7960
2001,0.7790
2002,0.7440
2003,0.6720
2004,0.6360
2005,0.6360
2006,0.6400
2007,0.6730


In [ ]:
# Page 142, Exhibit II Sheet 1, column 11
selected_claim_ratio = [0.75, 0.75, 0.75, 0.75, 0.75, 0.65, 0.65, 0.65, 0.65, 0.65]
selected_claim_ratio

[0.75, 0.75, 0.75, 0.75, 0.75, 0.65, 0.65, 0.65, 0.65, 0.65]

In [ ]:
# Page 142, Exhibit II Sheet 1, column 12
expected_claims = earned_premium * np.array(selected_claim_ratio).reshape(1, 1, -1, 1)
expected_claims

,2007
1998,"51,430,657"
1999,"51,408,736"
2000,"51,680,983"
2001,"54,408,716"
2002,"59,421,665"
2003,"56,318,302"
2004,"59,646,290"
2005,"61,174,953"
2006,"61,926,981"
2007,"61,864,556"


In [ ]:
# Page 143, Exhibit II Sheet 2, columns 1-2
print(friedland_us_industry_auto["Reported Claims"].latest_diagonal)
print("Total:", friedland_us_industry_auto["Reported Claims"].latest_diagonal.sum())

            2007
1998  47742304.0
1999  51185767.0
2000  54837929.0
2001  56299562.0
2002  58592712.0
2003  57565344.0
2004  56976657.0
2005  56786410.0
2006  54641339.0
2007  48853563.0
Total: 543481587.0


In [ ]:
# Page 143, Exhibit II Sheet 2, column 3
print(friedland_us_industry_auto["Paid Claims"].latest_diagonal)
print("Total:", friedland_us_industry_auto["Paid Claims"].latest_diagonal.sum())

            2007
1998  47644187.0
1999  51000534.0
2000  54533225.0
2001  55878421.0
2002  57807215.0
2003  55930654.0
2004  53774672.0
2005  50644994.0
2006  43606497.0
2007  27229969.0
Total: 498050368.0


In [ ]:
# Page 143, Exhibit II Sheet 2, column 4
print(np.round(expected_claims, 0))
print("Total:", np.round(expected_claims.sum(), 0))

            2007
1998  51430657.0
1999  51408736.0
2000  51680983.0
2001  54408716.0
2002  59421665.0
2003  56318302.0
2004  59646290.0
2005  61174953.0
2006  61926981.0
2007  61864556.0
Total: 569281839.0


In [ ]:
# Page 143, Exhibit II Sheet 2, column 5
case_outstanding = friedland_us_industry_auto["Reported Claims"].latest_diagonal - friedland_us_industry_auto["Paid Claims"].latest_diagonal
print(case_outstanding)
print("Total:", case_outstanding.sum())

            2007
1998     98117.0
1999    185233.0
2000    304704.0
2001    421141.0
2002    785497.0
2003   1634690.0
2004   3201985.0
2005   6141416.0
2006  11034842.0
2007  21623594.0
Total: 45431219.0
